# Источники данных и контроль качества

Блокнот связан со второй главой пособия и предназначен для первичной проверки структуры набора данных, типов признаков, пропусков и дубликатов.

## Цель и ожидаемые результаты

После выполнения блокнота читатель должен:

- понимать, какие сведения нужно зафиксировать о наборе данных до анализа;
- уметь проверять формат, типы, пропуски и дубликаты;
- уметь составлять краткий паспорт датасета;
- видеть связь между качеством исходной таблицы и качеством дальнейших выводов.

## Постановка задачи

На данном этапе требуется не обучение модели, а первичная верификация набора данных. Необходимо определить, насколько данные пригодны для последующего анализа, какие ограничения они содержат и какие этапы предобработки потребуются.

## Используемые библиотеки

| Библиотека | Назначение |
| --- | --- |
| pandas | чтение таблиц, проверка типов, пропусков и дубликатов |
| pathlib | работа с файловой структурой |
| IPython.display | аккуратный вывод паспортных сведений |
| src.display_utils | компактное представление выводов в учебном формате |

## Источник и описание данных

В установочном варианте блокнот демонстрирует методику контроля качества на небольшом учебном примере. Далее тот же порядок действий переносится на реальные датасеты кейсов курса.

## Пример структуры признаков

| Признак | Тип | Смысл |
| --- | --- | --- |
| timestamp | datetime | момент времени наблюдения |
| load_mw | float | активная мощность нагрузки |
| voltage_kv | float | напряжение в узле |
| feeder_id | category | идентификатор линии или фидера |

## Этапы анализа

### 1. Формирование учебного примера
Создается небольшая таблица с типовыми проблемами качества.

### 2. Проверка структуры и типов
Проверяется, как интерпретируются значения и какие столбцы требуют приведения типов.

### 3. Проверка пропусков и дубликатов
Оценивается полнота данных и наличие повторов записей.

### 4. Краткий паспорт набора данных
Формируется компактное текстовое описание проблем и последующих шагов.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.display_utils import display_callout, display_metric_table, display_markdown_list

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

sample = pd.DataFrame(
    {
        "timestamp": [
            "2025-01-10 08:00",
            "2025-01-10 08:15",
            "2025-01-10 08:15",
            "2025-01-10 08:30",
            None,
        ],
        "load_mw": [120.4, 121.1, 121.1, None, 125.7],
        "voltage_kv": [110.2, 109.9, 109.9, 110.1, 110.0],
        "feeder_id": ["F-01", "F-01", "F-01", "F-01", "F-01"],
    }
)
sample

## Интерпретация

Даже в небольшом учебном примере видны типовые проблемы: пропуски по времени и нагрузке, а также дублирующаяся запись. В реальных эксплуатационных данных аналогичные отклонения обычно встречаются чаще и в более сложной форме.

In [ ]:
sample["timestamp"] = pd.to_datetime(sample["timestamp"], errors="coerce")
profile = pd.DataFrame(
    {
        "dtype": sample.dtypes.astype(str),
        "missing_share": sample.isna().mean().round(3),
        "unique_values": sample.nunique(dropna=False),
    }
)
profile

## Интерпретация

Таблица профиля показывает, какие признаки уже имеют корректный тип, а какие еще нуждаются в преобразовании. Доля пропусков позволяет сразу выделить столбцы, требующие особого внимания на следующем этапе.

In [ ]:
duplicate_count = int(sample.duplicated().sum())
metrics = {
    "row_count": float(len(sample)),
    "column_count": float(sample.shape[1]),
    "duplicate_rows": float(duplicate_count),
    "max_missing_share": float(sample.isna().mean().max()),
}
display_metric_table(metrics, decimals=2)

## Интерпретация

Даже одна повторная запись в малой таблице искажает агрегаты. В больших наборах данных эффект может быть существенно сильнее, особенно если речь идет о расчетах средних нагрузок, частоты событий или распределений режимов.

In [ ]:
issues = []
if sample["timestamp"].isna().any():
    issues.append("присутствуют пропуски или некорректные значения времени")
if sample["load_mw"].isna().any():
    issues.append("присутствуют пропуски по измерению нагрузки")
if duplicate_count:
    issues.append("обнаружены дублирующиеся строки")

display_markdown_list("Выявленные проблемы качества", issues)
display_callout(
    "Паспорт учебного набора данных",
    "Набор содержит 5 записей, 4 признака и демонстрирует типовые проблемы качества: пропуски и дубликаты.",
    tone="warning",
)

## Итоговые выводы

- Контроль качества данных должен выполняться до статистического анализа и моделирования.
- Для первичной проверки достаточно структуры, типов, пропусков и дубликатов.
- Уже на этом этапе можно зафиксировать основные риски и план предобработки.

## Вопросы и задания для самостоятельной работы

1. Какие дополнительные проверки качества были бы полезны для временного ряда нагрузки?
2. Как изменится дальнейший анализ, если не удалить дубликаты?
3. Составьте паспорт для собственного набора данных или одного из открытых кейсов курса.

## Источники

1. [Глава 2 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/02-data-sources-and-quality.md)
2. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)